# Learned Bloom Filters — 2. The Standard Bloom Filter (baseline)

A Bloom filter is `m` bits plus `k` hash functions. To add x, set the `k` bits it hashes
to. To query x, check those bits: any 0 means **definitely not in S**; all 1 means
**probably in S**.

Useful formulas (n = number of keys):
- False-positive rate: $(1-e^{-kn/m})^k$
- Best number of hashes: $k^* = (m/n)\ln 2$
- Bits for a target FPR $p$: $m = -n\ln p / (\ln 2)^2$

We build it from scratch, **verify it matches the formula**, then compute the baseline
memory for our blocklist.

In [1]:
import math
import hashlib
import numpy as np


class BloomFilter:
    """A standard Bloom filter: m bits, k hash functions, never a false negative."""

    def __init__(self, m_bits, n_expected=None, k=None):
        self.m = int(max(1, m_bits))
        if k is None and n_expected:
            # Best number of hash functions for m bits and n keys is (m/n) * ln2
            k = round((self.m / n_expected) * math.log(2))
        self.k = int(max(1, k or 7))
        self.bits = np.zeros(self.m, dtype=np.uint8)

    def _positions(self, item):
        # Kirsch-Mitzenmacher trick: one MD5 -> two numbers -> k bit positions
        digest = hashlib.md5(item.encode("utf-8", "ignore")).digest()
        h1 = int.from_bytes(digest[:8], "little")
        h2 = int.from_bytes(digest[8:], "little") | 1
        return [(h1 + i * h2) % self.m for i in range(self.k)]

    def add(self, item):
        for p in self._positions(item):
            self.bits[p] = 1

    def add_all(self, items):
        for item in items:
            self.add(item)

    def __contains__(self, item):
        # All k bits set -> "probably in"; any bit is 0 -> "definitely not in"
        return all(self.bits[p] for p in self._positions(item))


def bf_theoretical_fpr(m, n, k):
    # Textbook false-positive rate of a Bloom filter
    return (1 - math.exp(-k * n / m)) ** k


def bf_bits_for_fpr(n, target_fpr):
    # Bits needed for n keys at a target FPR p:  m = -n*ln(p) / (ln2)^2
    if target_fpr <= 0:
        return 10 ** 9
    return math.ceil(-n * math.log(target_fpr) / (math.log(2) ** 2))

Verify our implementation: empirical false-positive rate should match the formula, and false negatives must always be zero.

In [2]:
# Insert 10,000 keys, then query 20,000 strangers at different memory sizes
keys = ["key_%d" % i for i in range(10000)]
strangers = ["stranger_%d" % i for i in range(20000)]

print("bits/key | k | empirical FPR | theory  | false negatives")
for bits_per_key in [6, 8, 10, 12]:
    bf = BloomFilter(bits_per_key * len(keys), n_expected=len(keys))
    bf.add_all(keys)

    false_neg = sum(key not in bf for key in keys)        # must be 0
    false_pos = np.mean([s in bf for s in strangers])
    theory = bf_theoretical_fpr(bf.m, len(keys), bf.k)

    print("   %2d    | %d |    %.4f     | %.4f  |   %d"
          % (bits_per_key, bf.k, false_pos, theory, false_neg))

bits/key | k | empirical FPR | theory  | false negatives
    6    | 4 |    0.0573     | 0.0561  |   0
    8    | 6 |    0.0228     | 0.0216  |   0


   10    | 7 |    0.0090     | 0.0082  |   0
   12    | 8 |    0.0032     | 0.0031  |   0


Now the baseline: how much memory a standard Bloom filter needs for our set of malicious URLs.

In [3]:
# Download the URL dataset (~195k URLs, labelled malicious=1 / benign=0)
import os
import ssl
import urllib.request
import pandas as pd

DATA_URL = (
    "https://raw.githubusercontent.com/incertum/cyber-matrix-ai/"
    "master/Malicious-URL-Detection-Deep-Learning/data/url_data_mega_deep_learning.csv"
)

os.makedirs("data", exist_ok=True)

if not os.path.exists("data/url_data.csv"):
    # Relax TLS just for this one download (some Colab images lag on CA certs)
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE
    request = urllib.request.Request(DATA_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(request, timeout=120, context=ctx) as response:
        with open("data/url_data.csv", "wb") as out:
            out.write(response.read())

print("Dataset ready.")

Dataset ready.


In [4]:
df = pd.read_csv("data/url_data.csv")
df.columns = [c.strip() for c in df.columns]
df = df.dropna().drop_duplicates("url").reset_index(drop=True)
n_keys = int(df.isMalicious.sum())

print("Set S =", n_keys, "malicious URLs")
print()
print("target FPR | standard Bloom filter")
for target in [3e-2, 1e-2, 3e-3, 1e-3, 3e-4, 1e-4]:
    kb = bf_bits_for_fpr(n_keys, target) / 8 / 1024
    print("  %.0e   |  %6.1f KB" % (target, kb))

Set S = 73367 malicious URLs

target FPR | standard Bloom filter
  3e-02   |    65.4 KB
  1e-02   |    85.8 KB
  3e-03   |   108.3 KB
  1e-03   |   128.8 KB
  3e-04   |   151.2 KB
  1e-04   |   171.7 KB


**Takeaway.** The standard filter needs ~86 KB at 1% FPR up to ~172 KB at 0.01% — and it can't use the fact that malicious URLs *look* different. That's the opening the oracle exploits next.